In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# To ignore warinings
import warnings
warnings.filterwarnings('ignore')

# **Import Dataset**

In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier

# **Read Dataset**

In [ ]:
df_train = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/train.csv",header = None)
df_test = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/test.csv",header = None)
df_trainLabels = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/trainLabels.csv",header = None)

# **Get Information About Data**

In [ ]:
print("=== Shape ===")
print(f"train: {df_train.shape}")
print(f"trainLabels: {df_trainLabels.shape}")
print(f"test: {df_test.shape}")

In [ ]:
df_train.head()

In [ ]:
df_trainLabels.head()

In [ ]:
df_test.head()

In [ ]:
df_train.info()

In [ ]:
df_train.describe()

In [ ]:
df_trainLabels.describe()

In [ ]:
df_test.describe()

# **Training Data**

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(df_train,df_trainLabels,test_size=0.33,random_state=42)

In [ ]:
print(X_train.shape,y_train.shape)
print(X_test.shape,y_test.shape)

# Variable Initialization

In [ ]:
# LogisticRegression

log_reg = LogisticRegression()
log_reg.fit(X_train,y_train)

y_pred = pd.Series(log_reg.predict(X_test)) 
log_res_acc = accuracy_score(y_test,y_pred)
print(f"LogisticRegression accuracy: {log_res_acc}")

# Define Grid Parameters

In [ ]:
# DecisionTreeClassifier

param_grid = [{
    'criterion': ['gini','entropy','log_loss'],
    'max_depth': [i for i in range(1,13)],
    'min_samples_split': [i for i in range(1,13)]
}]

des_tree = DecisionTreeClassifier()

clf = GridSearchCV(des_tree, param_grid)
clf.fit(X_train,y_train)

y_pred = pd.Series(clf.predict(X_test))

des_tree_acc = accuracy_score(y_test, y_pred)

print(f"DecisionTreeClassifier: {des_tree_acc}") 
print(clf.best_params_)

# Random Forest Model Initialization

In [ ]:
# RandomForestClassifier

rand_forst = RandomForestClassifier(n_estimators = 1000,criterion = 'gini',max_depth=7)
rand_forst.fit(X_train,y_train)

y_pred = pd.Series(rand_forst.predict(X_test))
rand_forst_acc = accuracy_score(y_test, y_pred)

print(f"RandomForestClassifier: {rand_forst_acc}")

# SVC Model Initialization

In [ ]:
# SVC

svc = SVC()
svc.fit(X_train,y_train)
y_pred = pd.Series(svc.predict(X_test))

svc_acc = accuracy_score(y_test, y_pred)
print(f"SVC: {svc_acc}")

# Gradient Boosting Model Initialization

In [ ]:
# GradientBoostingClassifier

gbc= GradientBoostingClassifier(random_state=0)
gbc.fit(X_train,y_train)
y_pred = pd.Series(gbc.predict(X_test))

gbc_acc = accuracy_score(y_test, y_pred)
print(f"GradientBoostingClassifier: {gbc_acc}")

# CatBoost Model Initialization

In [ ]:
# CatBoost

cbc = CatBoostClassifier(iterations=1000, depth=7, learning_rate=0.01, verbose=0,loss_function='Logloss', random_seed=42)
cbc.fit(X_train,y_train)
y_pred = pd.Series(cbc.predict(X_test))

cbc_acc = accuracy_score(y_test, y_pred)
print(f"CatBoostClassifier: {cbc_acc}")

# **Data visualization & Displaying K-Nearest Neighbors (KNN) Prediction Results**

In [ ]:
# KNN

knn_acc = 0 # max accuracy
mx_n = 0 # max_accuracy n_neighbours

n_arr = [] # list for n_neight
n_score = [] # list for accuracies

for n_neigh in range(1,50):
    
    knn = KNeighborsClassifier(n_neighbors=n_neigh)
    knn.fit(X_train, y_train)
    
    y_pred = pd.Series(knn.predict(X_test))
    
    n_acc = accuracy_score(y_test, y_pred)
    
    n_arr.append(n_neigh)
    n_score.append(n_acc)
    
    if n_acc > knn_acc:
        knn_acc = n_acc
        mx_n = n_neigh

print("=== KNN ===")
print(f"Max accuracy: {knn_acc}")
print(f"Best param: {mx_n}") 
print("===========")
sns.lineplot(x = n_arr,y = n_score)

In [ ]:
models = ['LogisticRegression','RandomForestClassifier','DecisionTreeClassifier','KNN','SVC','GradientBoosting','CatBoost']
models_acc = [log_res_acc,rand_forst_acc,des_tree_acc,knn_acc,svc_acc,gbc_acc,cbc_acc]

sns.barplot(y = models,x = models_acc)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=6)
knn.fit(df_train,df_trainLabels)

y_ans = pd.Series(knn.predict(df_test))

#cbc = CatBoostClassifier(iterations=1000, depth=7, learning_rate=0.01, verbose=0,loss_function='Logloss', random_seed=42)
#cbc.fit(df_train,df_trainLabels)

#y_ans = pd.Series(cbc.predict(df_test))

In [ ]:
y_ans

In [ ]:
d = {'Solution':y_ans,'Id': [i for i in range(1,9001)]}
ans = pd.DataFrame(data = d,dtype=np.int64)
ans

# Save DataFrame

In [ ]:
ans.to_csv('out.csv', index=False)